# 03. Data Preprocessing

## Objective

The goal of this notebook is to prepare the MovieLens 32M dataset
for the recommendation models.

We will:

- Clean and standardize movie metadata
- Process genres and tags
- Prepare the ratings data
- Analyze user and movie activity thresholds
- Create datasets for content-based and collaborative filtering
- Save the processed datasets for later modeling

In [4]:
import pandas as pd
import numpy as np

from pathlib import Path

In [5]:
PROJECT_DIR = Path.cwd().parent

MOVIELENS_DIR = PROJECT_DIR / "data" / "raw" / "ml-32m"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data:", MOVIELENS_DIR)
print("Processed data:", PROCESSED_DIR)

Raw data: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\raw\ml-32m
Processed data: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\processed


In [6]:
# Load Raw Datasets
movies = pd.read_csv(MOVIELENS_DIR / "movies.csv")

ratings = pd.read_csv(
    MOVIELENS_DIR / "ratings.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
        "timestamp": "int64"
    }
)

tags = pd.read_csv(
    MOVIELENS_DIR / "tags.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "tag": "string",
        "timestamp": "int64"
    }
)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [7]:
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)


Movies: (87585, 3)
Ratings: (32000204, 4)
Tags: (2000072, 4)


In [9]:
#Inspect movie columns and data types
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  87585 non-null  int64
 1   title    87585 non-null  str  
 2   genres   87585 non-null  str  
dtypes: int64(1), str(2)
memory usage: 5.2 MB


In [10]:
# Check for missing values in the movie metadata
movies.isna().sum()

movieId    0
title      0
genres     0
dtype: int64

In [11]:
# Check for duplicate movie IDs
movies["movieId"].duplicated().sum()

np.int64(0)

In [12]:
# Count duplicate movie titles
movies["title"].duplicated().sum()

np.int64(203)

In [13]:
# Count movies with no genre information
(movies["genres"] == "(no genres listed)").sum()

np.int64(7080)

In [14]:
# Replace the placeholder genre value with an empty string
movies["genres_clean"] = movies["genres"].replace(
    "(no genres listed)",
    ""
)

In [15]:
# Check the first few rows of the cleaned genre column
movies[["movieId", "genres", "genres_clean"]].head()

,movieId,genres,genres_clean
0,1,Adventure|Animation|Children|Comedy|Fantasy,Adventure|Animation|Children|Comedy|Fantasy
1,2,Adventure|Children|Fantasy,Adventure|Children|Fantasy
2,3,Comedy|Romance,Comedy|Romance
3,4,Comedy|Drama|Romance,Comedy|Drama|Romance
4,5,Comedy,Comedy


In [16]:
# Extract the four-digit release year from the end of each movie title
movies["release_year"] = (
    movies["title"]
    .str.extract(r"\((\d{4})\)$")[0]
    .astype("Int64")
)

In [17]:
# Count movies with a missing release year
movies["release_year"].isna().sum()

np.int64(771)

In [18]:
# Inspect movie titles where the release year could not be extracted
movies.loc[
    movies["release_year"].isna(),
    ["movieId", "title"]
].head(20)

,movieId,title
8465,25936,"Babe Ruth Story, The (1948)"
8604,26137,"Heroes of Telemark, The (1965)"
8912,26628,Jack's Back (1988)
9122,27189,After the Rain (Ame agaru) (1999)
9765,32497,Love Letter (1995)
9878,33092,Acts of Worship (2001)
9998,33755,Godzilla vs. Biollante (Gojira vs. Biorante) (...
10059,34057,Godzilla vs. Hedorah (Gojira tai Hedorâ) (Godz...
10643,43892,Le créateur (1999)
11496,52118,R.S.V.P. (2002)


In [19]:
# Extract a four-digit year appearing anywhere in the movie title
movies["release_year"] = (
    movies["title"]
    .str.extract(r"\((\d{4})\)")[0]
    .astype("Int64")
)

In [20]:
# Count movies with a missing release year after the improved extraction
movies["release_year"].isna().sum()

np.int64(615)

In [21]:
# Check the range and distribution of the cleaned release years
movies["release_year"].describe()

count        86970.0
mean     1995.354226
std        26.011089
min           1874.0
25%           1981.0
50%           2006.0
75%           2015.0
max           2023.0
Name: release_year, dtype: Float64

In [22]:
# Check for release years outside the expected range
movies.loc[
    (movies["release_year"] < 1800) |
    (movies["release_year"] > 2023),
    ["movieId", "title", "release_year"]
]

,movieId,title,release_year


In [23]:
# Create the cleaned movie metadata table
movies_clean = movies[
    ["movieId", "title", "genres_clean", "release_year"]
].copy()

movies_clean.head()

,movieId,title,genres_clean,release_year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II (1995),Comedy,1995


In [24]:
# Check the ratings data types and non-null values
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int32  
 1   movieId    int32  
 2   rating     float32
 3   timestamp  int64  
dtypes: float32(1), int32(2), int64(1)
memory usage: 610.4 MB


In [25]:
# Check for missing values in the ratings data
ratings.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [26]:
# Check for duplicate rating records
ratings.duplicated().sum()

np.int64(0)

In [27]:
# Check the unique rating values
sorted(ratings["rating"].unique())

[np.float32(0.5),
 np.float32(1.0),
 np.float32(1.5),
 np.float32(2.0),
 np.float32(2.5),
 np.float32(3.0),
 np.float32(3.5),
 np.float32(4.0),
 np.float32(4.5),
 np.float32(5.0)]

In [28]:
# Check for ratings that reference movies missing from the movie metadata
missing_movie_ratings = ~ratings["movieId"].isin(movies_clean["movieId"])

print("Ratings with missing movie metadata:", missing_movie_ratings.sum())

Ratings with missing movie metadata: 0


In [29]:
# Compare how many movies and ratings remain under different minimum-rating thresholds
movie_rating_counts = ratings.groupby("movieId").size()

thresholds = [5, 10, 20, 50, 100]

threshold_results = []

for threshold in thresholds:
    eligible_movies = movie_rating_counts[
        movie_rating_counts >= threshold
    ].index
    
    eligible_ratings = ratings[
        ratings["movieId"].isin(eligible_movies)
    ]
    
    threshold_results.append({
        "min_ratings_per_movie": threshold,
        "movies": len(eligible_movies),
        "ratings": len(eligible_ratings),
        "rating_coverage_pct": len(eligible_ratings) / len(ratings) * 100
    })

pd.DataFrame(threshold_results)

,min_ratings_per_movie,movies,ratings,rating_coverage_pct
0,5,43884,31921467,99.753948
1,10,31961,31842705,99.507819
2,20,23350,31725920,99.142868
3,50,16034,31498689,98.432776
4,100,12191,31228923,97.589762


In [30]:
# Compare how many users and ratings remain under different minimum-rating thresholds
user_rating_counts = ratings.groupby("userId").size()

user_thresholds = [20, 50, 100, 200, 500]

user_threshold_results = []

for threshold in user_thresholds:
    eligible_users = user_rating_counts[
        user_rating_counts >= threshold
    ].index
    
    eligible_ratings = ratings[
        ratings["userId"].isin(eligible_users)
    ]
    
    user_threshold_results.append({
        "min_ratings_per_user": threshold,
        "users": len(eligible_users),
        "ratings": len(eligible_ratings),
        "rating_coverage_pct": len(eligible_ratings) / len(ratings) * 100
    })

pd.DataFrame(user_threshold_results)

,min_ratings_per_user,users,ratings,rating_coverage_pct
0,20,200948,32000204,100.000000
1,50,128344,29703484,92.822796
2,100,80675,26350042,82.343356
3,200,42197,20992152,65.600057
4,500,12603,11928570,37.276544


Important modeling decision

We should not automatically filter users at 100 or 500 ratings.

For our project, we can use different thresholds for different recommenders:

Content-based: no user filtering needed.
Collaborative filtering: potentially use a minimum user/movie activity threshold to improve the quality and efficiency of the interaction matrix.
Also-watched: focus on meaningful interactions rather than requiring extremely active users.
Hybrid: combine the models without unnecessarily throwing away useful information.

In [31]:
# Compare combined user and movie activity thresholds
combined_thresholds = [
    (20, 5),
    (50, 5),
    (50, 10),
    (100, 10),
    (100, 20),
    (100, 50)
]

combined_results = []

for user_threshold, movie_threshold in combined_thresholds:
    eligible_users = user_rating_counts[
        user_rating_counts >= user_threshold
    ].index

    eligible_movies = movie_rating_counts[
        movie_rating_counts >= movie_threshold
    ].index

    filtered_ratings = ratings[
        ratings["userId"].isin(eligible_users) &
        ratings["movieId"].isin(eligible_movies)
    ]

    combined_results.append({
        "min_ratings_per_user": user_threshold,
        "min_ratings_per_movie": movie_threshold,
        "users": filtered_ratings["userId"].nunique(),
        "movies": filtered_ratings["movieId"].nunique(),
        "ratings": len(filtered_ratings),
        "rating_coverage_pct": len(filtered_ratings) / len(ratings) * 100
    })

pd.DataFrame(combined_results)

,min_ratings_per_user,min_ratings_per_movie,users,movies,ratings,rating_coverage_pct
0,20,5,200948,43884,31921467,99.753948
1,50,5,128344,43884,29626133,92.581075
2,50,10,128344,31961,29548646,92.338930
3,100,10,80675,31961,26197986,81.868184
4,100,20,80675,23350,26085590,81.516949
5,100,50,80675,16034,25868292,80.837897


# Create the collaborative-filtering dataset

In [32]:
# Create the filtered ratings dataset for collaborative filtering
cf_user_threshold = 50
cf_movie_threshold = 10

eligible_cf_users = user_rating_counts[
    user_rating_counts >= cf_user_threshold
].index

eligible_cf_movies = movie_rating_counts[
    movie_rating_counts >= cf_movie_threshold
].index

ratings_cf = ratings[
    ratings["userId"].isin(eligible_cf_users) &
    ratings["movieId"].isin(eligible_cf_movies)
].copy()

print("Collaborative filtering ratings:", len(ratings_cf))
print("Users:", ratings_cf["userId"].nunique())
print("Movies:", ratings_cf["movieId"].nunique())

Collaborative filtering ratings: 29548646
Users: 128344
Movies: 31961


In [33]:
# Verify that every user and movie in the collaborative filtering dataset meets the selected thresholds
cf_user_counts = ratings_cf.groupby("userId").size()
cf_movie_counts = ratings_cf.groupby("movieId").size()

print("Minimum ratings per user:", cf_user_counts.min())
print("Minimum ratings per movie:", cf_movie_counts.min())

Minimum ratings per user: 15
Minimum ratings per movie: 5


In [34]:
# Iteratively filter users and movies until both activity thresholds are satisfied
ratings_cf = ratings.copy()

while True:
    previous_size = len(ratings_cf)

    user_counts = ratings_cf.groupby("userId").size()
    eligible_users = user_counts[user_counts >= 50].index

    ratings_cf = ratings_cf[
        ratings_cf["userId"].isin(eligible_users)
    ]

    movie_counts = ratings_cf.groupby("movieId").size()
    eligible_movies = movie_counts[movie_counts >= 10].index

    ratings_cf = ratings_cf[
        ratings_cf["movieId"].isin(eligible_movies)
    ]

    if len(ratings_cf) == previous_size:
        break

print("Collaborative filtering ratings:", len(ratings_cf))
print("Users:", ratings_cf["userId"].nunique())
print("Movies:", ratings_cf["movieId"].nunique())

Collaborative filtering ratings: 29542223
Users: 128255
Movies: 31701


In [35]:
# Verify that the final collaborative filtering dataset satisfies both activity thresholds
final_user_counts = ratings_cf.groupby("userId").size()
final_movie_counts = ratings_cf.groupby("movieId").size()

print("Minimum ratings per user:", final_user_counts.min())
print("Minimum ratings per movie:", final_movie_counts.min())

Minimum ratings per user: 50
Minimum ratings per movie: 10


# Tags analysis and preparation

In [36]:
# Check the quality and coverage of the movie tags before processing them
print("Total tags:", len(tags))
print("Unique tag texts:", tags["tag"].nunique())
print("Movies with tags:", tags["movieId"].nunique())
print("Users who created tags:", tags["userId"].nunique())
print("Missing tags:", tags["tag"].isna().sum())

Total tags: 2000072
Unique tag texts: 140979
Movies with tags: 51323
Users who created tags: 15848
Missing tags: 17


In [37]:
# Inspect a sample of movie tags to identify formatting and cleaning issues
tags[["movieId", "tag"]].sample(20, random_state=42)

,movieId,tag
1602011,48043,atmospheric
1831526,50594,art film
777298,4942,machete
1956230,74458,shocking ending
1759108,122916,Thor
1432474,162738,road trip
385481,185029,survival
1723387,91500,based on a book
819177,6440,lynchian
1958584,147,drugs


In [38]:
# Clean the tag text by removing missing values, extra spaces, and capitalization differences
tags_clean = tags.dropna(subset=["tag"]).copy()

tags_clean["tag_clean"] = (
    tags_clean["tag"]
    .str.strip()
    .str.lower()
)

tags_clean[["movieId", "tag", "tag_clean"]].head(20)

,movieId,tag,tag_clean
0,26479,Kevin Kline,kevin kline
1,79592,misogyny,misogyny
2,247150,acrophobia,acrophobia
3,2174,music,music
4,2174,weird,weird
5,8623,Steve Martin,steve martin
6,5766,the killls and the score,the killls and the score
7,7451,bullying,bullying
8,7451,clique,clique
9,7451,coming of age,coming of age


In [39]:
# Check for empty or missing values in the cleaned tag column
print("Missing cleaned tags:", tags_clean["tag_clean"].isna().sum())
print("Empty cleaned tags:", (tags_clean["tag_clean"] == "").sum())

Missing cleaned tags: 0
Empty cleaned tags: 6


In [41]:
# Remove tags that contain no text after cleaning
tags_clean = tags_clean[
    tags_clean["tag_clean"] != ""
].copy()

print("Clean tags:", len(tags_clean))
print("Empty cleaned tags:", (tags_clean["tag_clean"] == "").sum())

Clean tags: 2000049
Empty cleaned tags: 0


In [42]:
# Measure how many cleaned tags are associated with each movie
tags_per_movie = tags_clean.groupby("movieId").size()

tags_per_movie.describe()

count    51323.000000
mean        38.969838
std        160.935045
min          1.000000
25%          2.000000
50%          5.000000
75%         16.000000
max       6697.000000
dtype: float64

In [43]:
# Find the most frequently used cleaned tags across all movies
tag_frequency = (
    tags_clean["tag_clean"]
    .value_counts()
    .head(30)
)

tag_frequency

tag_clean
sci-fi                11457
atmospheric           10088
action                 9344
comedy                 9275
funny                  8088
surreal                7480
visually appealing     7253
based on a book        6733
twist ending           6600
romance                6229
dark comedy            6153
thought-provoking      6003
dystopia               5713
violence               5483
cinematography         5429
murder                 5238
social commentary      5201
fantasy                5154
thriller               5050
quirky                 4954
stylized               4948
adventure              4860
psychology             4847
dark                   4825
classic                4741
time travel            4740
revenge                4724
coming of age          4494
horror                 4441
drama                  4407
Name: count, dtype: int64[pyarrow]

In [44]:
# Combine all cleaned tags belonging to each movie into one text field
movie_tags = (
    tags_clean
    .groupby("movieId")["tag_clean"]
    .apply(" ".join)
    .reset_index(name="tags_text")
)

movie_tags.head()

,movieId,tags_text
0,1,children disney animation children disney disn...
1,2,robin williams fantasy robin williams time tra...
2,3,comedinha de velhinhos engraãƒâ§ada comedinha ...
3,4,characters slurs based on novel or book chick ...
4,5,fantasy pregnancy remake family steve martin s...


### Data Quality Note: Tag Encoding

A small number of tags contain encoding artifacts, including some corrupted
non-ASCII characters. An initial inspection and attempted automated repair
were considered, but the affected records represented only a very small
portion of the tag dataset.

Because automated repair could incorrectly modify valid Unicode text, the
original cleaned tag values were retained rather than applying aggressive
encoding corrections.

This preserves the original information while avoiding unnecessary
alteration of the dataset.


In [53]:
# Check how many movies have tag information
print("Movies with tags:", movie_tags["movieId"].nunique())
print("Total movies:", movies_clean["movieId"].nunique())

Movies with tags: 51323
Total movies: 87585


In [54]:
# Add aggregated tag information to the cleaned movie metadata
movies_content = movies_clean.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies_content.head()

,movieId,title,genres_clean,release_year,tags_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,children disney animation children disney disn...
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,robin williams fantasy robin williams time tra...
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,comedinha de velhinhos engraãƒâ§ada comedinha ...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,characters slurs based on novel or book chick ...
4,5,Father of the Bride Part II (1995),Comedy,1995,fantasy pregnancy remake family steve martin s...


In [55]:
# Check how many movies do not have tag information
print("Movies without tags:", movies_content["tags_text"].isna().sum())
print("Movies with tags:", movies_content["tags_text"].notna().sum())

Movies without tags: 36262
Movies with tags: 51323


In [56]:
# Replace missing tag text with an empty string
movies_content["tags_text"] = movies_content["tags_text"].fillna("")

movies_content[["movieId", "genres_clean", "tags_text"]].head()

,movieId,genres_clean,tags_text
0,1,Adventure|Animation|Children|Comedy|Fantasy,children disney animation children disney disn...
1,2,Adventure|Children|Fantasy,robin williams fantasy robin williams time tra...
2,3,Comedy|Romance,comedinha de velhinhos engraãƒâ§ada comedinha ...
3,4,Comedy|Drama|Romance,characters slurs based on novel or book chick ...
4,5,Comedy,fantasy pregnancy remake family steve martin s...


In [57]:
# Combine genres and tags into one text field for content-based modeling
movies_content["content_text"] = (
    movies_content["genres_clean"]
    .str.replace("|", " ", regex=False)
    + " "
    + movies_content["tags_text"]
)

movies_content[["movieId", "title", "content_text"]].head()

,movieId,title,content_text
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy ch...
1,2,Jumanji (1995),Adventure Children Fantasy robin williams fant...
2,3,Grumpier Old Men (1995),Comedy Romance comedinha de velhinhos engraãƒâ...
3,4,Waiting to Exhale (1995),Comedy Drama Romance characters slurs based on...
4,5,Father of the Bride Part II (1995),Comedy fantasy pregnancy remake family steve m...


In [58]:
# Convert the combined content text to lowercase for consistent text processing
movies_content["content_text"] = movies_content["content_text"].str.lower()

movies_content[["movieId", "title", "content_text"]].head()

,movieId,title,content_text
0,1,Toy Story (1995),adventure animation children comedy fantasy ch...
1,2,Jumanji (1995),adventure children fantasy robin williams fant...
2,3,Grumpier Old Men (1995),comedy romance comedinha de velhinhos engraãƒâ...
3,4,Waiting to Exhale (1995),comedy drama romance characters slurs based on...
4,5,Father of the Bride Part II (1995),comedy fantasy pregnancy remake family steve m...


In [59]:
# Check for missing or empty content text
print("Missing content:", movies_content["content_text"].isna().sum())
print(
    "Empty content:",
    (movies_content["content_text"].str.strip() == "").sum()
)


Missing content: 0
Empty content: 4420


### Content Data Quality Note

After combining genres and tags, 4,420 movies have no available genre or
tag information and therefore have an empty `content_text` field.

These movies are retained in the dataset rather than removed. They may
still contain useful information for collaborative-filtering models, but
the content-based model will have limited information for recommending
similar movies for them.


In [60]:
# Verify that all movies are still present after content preprocessing
print("Movies in content dataset:", len(movies_content))
print("Unique movie IDs:", movies_content["movieId"].nunique())

Movies in content dataset: 87585
Unique movie IDs: 87585


In [61]:
# Save the processed movie content dataset for the recommendation models
movies_content.to_csv(
    PROCESSED_DIR / "movies_content.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "movies_content.csv")

Saved: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\processed\movies_content.csv


In [62]:
# Save the filtered ratings dataset for collaborative filtering
ratings_cf.to_csv(
    PROCESSED_DIR / "ratings_cf.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "ratings_cf.csv")

Saved: c:\Users\moham\Desktop\Netflix Movie recomendation system\data\processed\ratings_cf.csv


In [63]:
# Verify the final processed datasets and their key dimensions
print("Content dataset:")
print("Rows:", len(movies_content))
print("Unique movies:", movies_content["movieId"].nunique())

print("\nCollaborative filtering dataset:")
print("Ratings:", len(ratings_cf))
print("Users:", ratings_cf["userId"].nunique())
print("Movies:", ratings_cf["movieId"].nunique())

Content dataset:
Rows: 87585
Unique movies: 87585

Collaborative filtering dataset:
Ratings: 29542223
Users: 128255
Movies: 31701


## Conclusion

The MovieLens 32M dataset has been successfully preprocessed for the
recommendation-system project.

The preprocessing produced two main datasets:

* **Movie content dataset:** 87,585 movies with cleaned genres, release
  years, and aggregated tag information.
* **Collaborative filtering dataset:** 29,542,223 ratings from 128,255 users
  across 31,701 movies after applying iterative activity thresholds.

For content-based recommendation, genres and user-generated tags were
combined into a unified text representation. Movies without available
content information were retained rather than removed.

For collaborative filtering, a minimum activity threshold of 50 ratings
per user and 10 ratings per movie was applied iteratively to reduce sparsity
while retaining a large portion of the original rating data.

The processed datasets have been saved in the `data/processed/` directory
and are ready for the recommendation models.

### Next Step

The next notebook will develop the **content-based recommender** using
TF-IDF and cosine similarity to identify movies with similar content.
